# Pandas Read & Write Data Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. A CSV is just text.** WITHOUT pandas: use `pathlib` to make sure a `sample_data/` folder exists next to your notebook (`mkdir(exist_ok=True)`), then use the standard-library `csv` module to write `exercise_books.csv` containing header `title,pages` and rows Dune 412, Foundation 255, I Robot 224. Print the raw file contents. Finish by deleting your scratch file with `books_path.unlink()` so the folder stays clean.

In [ ]:
import csv
from pathlib import Path

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)                     # safe to re-run
books_path = out_dir / "exercise_books.csv"

rows = [
    ["title", "pages"],
    ["Dune", 412],
    ["Foundation", 255],
    ["I Robot", 224],
]
with open(books_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)

print(books_path.read_text(encoding="utf-8"))    # plain text -- no magic
books_path.unlink()                              # tidy up our scratch file

**2. Read it back.** Recreate `exercise_books.csv` exactly as in Q1 (same three books), then load it with `pd.read_csv`. Print `head(2)`, the `shape`, and `dtypes` — what type did pandas GUESS for `pages`? Delete the file again at the end.

In [ ]:
import csv
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
books_path = out_dir / "exercise_books.csv"

with open(books_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["title", "pages"],
        ["Dune", 412],
        ["Foundation", 255],
        ["I Robot", 224],
    ])

df = pd.read_csv(books_path)
print(df.head(2))
print()
print("shape:", df.shape)    # (3, 2)
print(df.dtypes)             # pages guessed as a number -- good guess
books_path.unlink()

**3. Load less data.** Write `exercise_cities.csv` with header `city,population_millions` and five rows: Dhaka 22.5, Chattogram 5.4, Khulna 2.0, Rajshahi 3.3, Sylhet 3.9. Read back ONLY the `population_millions` column and ONLY the first two rows (`usecols=` + `nrows=`). Print the slim result and clean up.

In [ ]:
import csv
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
cities_path = out_dir / "exercise_cities.csv"

with open(cities_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["city", "population_millions"],
        ["Dhaka", 22.5],
        ["Chattogram", 5.4],
        ["Khulna", 2.0],
        ["Rajshahi", 3.3],
        ["Sylhet", 3.9],
    ])

slim = pd.read_csv(cities_path, usecols=["population_millions"], nrows=2)
print(slim)
cities_path.unlink()

## Part 2 — Practice

**4. The missing header.** Sensor-style files carry numbers only, NO title line. Write `exercise_readings.csv` with rows Rafi 92, Mim 84, Tanvir 77 and no header. Read it plainly first and observe what happened to poor Rafi; then fix it with `header=None, names=["name", "score"]`. Print both attempts and clean up.

In [ ]:
import csv
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
readings_path = out_dir / "exercise_readings.csv"

with open(readings_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([["Rafi", 92], ["Mim", 84], ["Tanvir", 77]])

oops = pd.read_csv(readings_path)      # first DATA row became the headers!
print(oops)
print()
fixed = pd.read_csv(readings_path, header=None, names=["name", "score"])
print(fixed)
readings_path.unlink()

**5. Leading zeros must survive.** Write `exercise_orders.csv` with header `order_id,amount` and rows "0012" 450 and "0099" 120 (quote the IDs so the zeros reach the file!). Load it twice: once lazily, once with `dtype={"order_id": str}`. Print both tables and their dtypes — what did the lazy guess cost us?

In [ ]:
import csv
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
orders_path = out_dir / "exercise_orders.csv"

with open(orders_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["order_id", "amount"],
        ["0012", 450],
        ["0099", 120],
    ])

lazy = pd.read_csv(orders_path)                             # guesses a number
forced = pd.read_csv(orders_path, dtype={"order_id": str})  # kept as text

print(lazy)
print(lazy.dtypes)      # order_id guessed numeric -> zeros LOST
print()
print(forced)
print(forced.dtypes)    # order_id kept as text -> "0012" survives
orders_path.unlink()

**6. Junk markers.** Write `exercise_survey.csv` with header `respondent,age` and rows r1 29, r2 ?, r3 missing. Read it twice: plainly, then with `na_values=["?", "missing"]`. Print the dtypes of BOTH reads, the cleaned table, and how many ages are missing via `isna().sum()`. Clean up.

In [ ]:
import csv
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
survey_path = out_dir / "exercise_survey.csv"

with open(survey_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["respondent", "age"],
        ["r1", 29],
        ["r2", "?"],
        ["r3", "missing"],
    ])

default_read = pd.read_csv(survey_path)               # junk strings -> TEXT column
cleaner_read = pd.read_csv(survey_path, na_values=["?", "missing"])

print(default_read.dtypes)
print(cleaner_read)
print("missing ages:", cleaner_read["age"].isna().sum())   # 2
survey_path.unlink()

**7. Fetch by ID directly.** Write `exercise_students.csv` with header `student_id,name,score` and rows S001 Sarah 88, S002 Rafi 92, S003 Nabila 79. Load it with `index_col="student_id"`, print the index as a list, then fetch the whole row labeled `"S002"` with `.loc`. Clean up afterwards.

In [ ]:
import csv
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
students_path = out_dir / "exercise_students.csv"

with open(students_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([
        ["student_id", "name", "score"],
        ["S001", "Sarah", 88],
        ["S002", "Rafi", 92],
        ["S003", "Nabila", 79],
    ])

by_id = pd.read_csv(students_path, index_col="student_id")
print(by_id.index.tolist())    # IDs, not 0..2
print()
print(by_id.loc["S002"])
students_path.unlink()

## Part 3 — Challenge

**8. The ghost column.** Build this table inline: `pd.DataFrame({"product": ["pen", "book"], "price": [25, 120]})`. Save it TWICE to CSV: once without thinking (`to_csv(path)`) and once properly (`index=False`). Read both back and print them. Which file grew an `Unnamed: 0` ghost column — and what was inside it? Clean up both files.

In [ ]:
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
df = pd.DataFrame({"product": ["pen", "book"], "price": [25, 120]})

sloppy_path = out_dir / "exercise_sloppy.csv"
clean_path = out_dir / "exercise_clean.csv"
df.to_csv(sloppy_path)                 # index silently included
df.to_csv(clean_path, index=False)     # data only

print(pd.read_csv(sloppy_path))        # ghost column = the old row labels
print()
print(pd.read_csv(clean_path))
sloppy_path.unlink()
clean_path.unlink()

**9. JSON round trip.** Inline-build a three-row pets table: `name` Rex/Whiskers/Bubbles, `sound` woof/meow/blub. Save it as JSON with `orient="records"` and `indent=2`, print the raw JSON text, read it back with `pd.read_json(..., orient="records")`, and verify in ONE printed boolean that columns and shape survived. Clean up.

In [ ]:
from pathlib import Path
import pandas as pd

out_dir = Path("sample_data")
out_dir.mkdir(exist_ok=True)
pets_path = out_dir / "exercise_pets.json"

pets = pd.DataFrame({
    "name":  ["Rex", "Whiskers", "Bubbles"],
    "sound": ["woof", "meow", "blub"],
})
pets.to_json(pets_path, orient="records", indent=2)

print(pets_path.read_text(encoding="utf-8"))    # rows became objects in a list
rebuilt = pd.read_json(pets_path, orient="records")
print()
print(rebuilt)
print("round trip ok:",
      list(rebuilt.columns) == list(pets.columns) and rebuilt.shape == pets.shape)
pets_path.unlink()